# Million-parameter Humanoid optimization with MJX and ENNx

This notebook directly optimizes a roughly one-million-parameter JAX policy from scalar MuJoCo task rewards. MJX runs batched Humanoid simulation on the T4, while CUDA-resident ENNx generates, scores, and selects int4 policy perturbations without gradients.

## 1. Prepare a T4 runtime

Select **Runtime > Change runtime type > T4 GPU**. The binary ENNx wheel is compiled for T4 `sm_75`; no Rust or CUDA compiler toolchain runs in this notebook.

In [ ]:
import os
import subprocess
import sys
from importlib import metadata
from pathlib import Path

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_DEFAULT_MATMUL_PRECISION"] = "highest"
os.environ["MUJOCO_GL"] = "egl"
assert sys.version_info[:2] == (3, 12)

numpy_version = metadata.version("numpy")
scipy_version = metadata.version("scipy")
constraints = Path("/tmp/ennx-colab-constraints.txt")
constraints.write_text(
    f"numpy=={numpy_version}\nscipy=={scipy_version}\n", encoding="utf-8"
)
print(f"preserving numpy={numpy_version} scipy={scipy_version}")

CUDA_WHEEL = (
    "https://github.com/Kvutza/ennx/releases/download/cuda-v0.1.5/"
    "ennx-0.1.5%2Bcuda75-cp312-cp312-manylinux_2_28_x86_64.whl"
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--constraint",
        str(constraints),
        "mujoco==3.6.0",
        "mujoco-mjx==3.6.0",
        "mediapy>=1.2",
    ],
    check=True,
)
assert metadata.version("numpy") == numpy_version
assert metadata.version("scipy") == scipy_version
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-deps",
        CUDA_WHEEL,
    ],
    check=True,
)

In [ ]:
import json
import platform
from pathlib import Path
from urllib.request import urlretrieve

import cupy as cp
import jax
import jax.numpy as jnp
import mediapy as media
import mujoco
import numpy as np
from mujoco import mjx

from ennx.experimental import TurboSearch

gpu_devices = [device for device in jax.devices() if device.platform == "gpu"]
runtime = {
    "python": platform.python_version(),
    "jax": jax.__version__,
    "mujoco": mujoco.__version__,
    "devices": [str(device) for device in jax.devices()],
}
print(json.dumps(runtime, indent=2))
assert gpu_devices, "JAX did not discover the T4"
GPU = gpu_devices[0]

## 2. Load the pinned Humanoid model

The MJCF model and its three visual includes are fetched from the MuJoCo Playground `v0.2.0` release. Simulation itself uses `mujoco.mjx` directly.

In [ ]:
MODEL_ROOT = Path("/content/mjx_humanoid")
MODEL_BASE = (
    "https://raw.githubusercontent.com/google-deepmind/"
    "mujoco_playground/v0.2.0/mujoco_playground/_src/"
    "dm_control_suite/xmls/"
)
MODEL_FILES = (
    "humanoid.xml",
    "common/skybox.xml",
    "common/visual.xml",
    "common/materials.xml",
)
for relative in MODEL_FILES:
    target = MODEL_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        urlretrieve(MODEL_BASE + relative, target)

SIM_DT = 0.005
SUBSTEPS = 5
CTRL_DT = SIM_DT * SUBSTEPS
mj_model = mujoco.MjModel.from_xml_path(str(MODEL_ROOT / "humanoid.xml"))
mj_model.opt.timestep = SIM_DT
mjx_model = mjx.put_model(mj_model, impl="jax")
data_seed = mjx.make_data(mj_model, impl="jax")
torso_id = mj_model.body("torso").id
print(f"nq={mj_model.nq} nv={mj_model.nv} actions={mj_model.nu}")

## 3. Define the MJX task

The policy observes root-relative position and velocity. Reward combines standing, upright posture, forward velocity, and a small control penalty. Episodes stop after a fall or a non-finite state.

In [ ]:
def observe(data):
    return jnp.concatenate((data.qpos[2:], data.qvel))

def reset_env(key):
    qpos_key, qvel_key = jax.random.split(key)
    qpos = jnp.asarray(mj_model.qpos0)
    qpos = qpos + 0.005 * jax.random.uniform(
        qpos_key, qpos.shape, minval=-1.0, maxval=1.0
    )
    qvel = 0.005 * jax.random.normal(qvel_key, (mj_model.nv,))
    data = data_seed.replace(qpos=qpos, qvel=qvel, ctrl=jnp.zeros(mj_model.nu))
    return mjx.forward(mjx_model, data)

def step_env(data, action):
    action = jnp.clip(action, -1.0, 1.0)
    old_x = data.qpos[0]

    def physics_step(carry, unused):
        del unused
        carry = carry.replace(ctrl=action)
        return mjx.step(mjx_model, carry), None

    data, _ = jax.lax.scan(physics_step, data, None, length=SUBSTEPS)
    height = data.xpos[torso_id, 2]
    upright = data.xmat[torso_id, 2, 2]
    standing = jnp.clip((height - 0.75) / 0.65, 0.0, 1.0)
    posture = jnp.clip((upright + 0.1) / 1.1, 0.0, 1.0)
    velocity = (data.qpos[0] - old_x) / CTRL_DT
    control = 0.002 * jnp.mean(jnp.square(action))
    reward = standing * posture * (1.0 + 0.25 * jnp.tanh(velocity)) - control
    finite = jnp.all(jnp.isfinite(data.qpos)) & jnp.all(jnp.isfinite(data.qvel))
    done = (~finite) | (height < 0.65) | (height > 2.25)
    return data, reward, done

sample_data = jax.jit(reset_env)(jax.random.PRNGKey(0))
OBS_SIZE = int(observe(sample_data).size)
ACTION_SIZE = int(mj_model.nu)
print(f"observation={OBS_SIZE} action={ACTION_SIZE} dt={CTRL_DT}")

## 4. Create the million-parameter JAX policy

The wide random features and small output initialization provide a stable near-zero-action center while exposing a genuinely high-dimensional search surface. ENNx receives no derivatives from this policy or the simulator.

In [ ]:
HIDDEN_SIZE = 2048
BOTTLENECK_SIZE = 416

def init_policy(key):
    keys = jax.random.split(key, 3)

    def make_layer(layer_key, inputs, outputs, scale=1.0):
        limit = scale * jnp.sqrt(6.0 / (inputs + outputs))
        weight = jax.random.uniform(
            layer_key, (inputs, outputs), minval=-limit, maxval=limit
        )
        return weight, jnp.zeros(outputs, dtype=jnp.float32)

    first = make_layer(keys[0], OBS_SIZE, HIDDEN_SIZE)
    second = make_layer(keys[1], HIDDEN_SIZE, BOTTLENECK_SIZE)
    output = make_layer(keys[2], BOTTLENECK_SIZE, ACTION_SIZE, scale=0.05)
    return first + second + output

def apply_policy(params, obs):
    w1, b1, w2, b2, w3, b3 = params
    hidden = jax.nn.tanh(obs @ w1 + b1)
    hidden = jax.nn.tanh(hidden @ w2 + b2)
    return jax.nn.tanh(hidden @ w3 + b3)

float_params = tuple(
    jax.device_put(value, GPU) for value in init_policy(jax.random.PRNGKey(7))
)
PARAMETER_COUNT = sum(int(value.size) for value in float_params)
assert 900_000 <= PARAMETER_COUNT <= 1_000_000
print(f"policy_parameters={PARAMETER_COUNT:,}")

## 5. Compile batched task evaluation

Eight deterministic initial conditions are evaluated in parallel. The returned objective is mean reward per control step, which remains on a stable scale for ENNx posterior scoring.

In [ ]:
EVAL_ENVS = 8
ROLLOUT_STEPS = 256
EVAL_KEYS = jax.random.split(jax.random.PRNGKey(23), EVAL_ENVS)

@jax.jit
def score_policy(params):
    data = jax.vmap(reset_env)(EVAL_KEYS)
    active = jnp.ones(EVAL_ENVS, dtype=jnp.float32)
    totals = jnp.zeros(EVAL_ENVS, dtype=jnp.float32)

    def rollout_step(carry, unused):
        del unused
        data, active, totals = carry
        obs = jax.vmap(observe)(data)
        action = apply_policy(params, obs)
        data, reward, done = jax.vmap(step_env)(data, action)
        totals = totals + active * reward
        active = active * (~done).astype(jnp.float32)
        return (data, active, totals), None

    (_, _, totals), _ = jax.lax.scan(
        rollout_step, (data, active, totals), None, length=ROLLOUT_STEPS
    )
    values = totals / ROLLOUT_STEPS
    return jnp.mean(values), jnp.std(values)

@jax.jit
def score_batch(params):
    return jax.vmap(score_policy)(params)

float_reward, float_std = score_policy(float_params)
float_reward.block_until_ready()
print(f"float_reward={float(float_reward):.5f} std={float(float_std):.5f}")

## 6. Encode the policy as structured int4 leaves

Each tensor is divided into blocks of at most 16,384 parameters. Blocks keep separate quantization scales while tensor-normalized metric weights prevent the two large matrices from erasing the smaller output layer.

In [ ]:
ZERO_POINT = 8
BLOCK_SIZE = 16_384

def pack_codes(codes):
    codes = np.asarray(codes, dtype=np.uint8).reshape(-1)
    if codes.size % 2:
        codes = np.pad(codes, (0, 1))
    return codes[0::2] | (codes[1::2] << np.uint8(4))

def encode_params(params):
    rows = []
    leaves = []
    specs = []
    element_offset = 0
    byte_offset = 0
    for value in params:
        host = np.asarray(value, dtype=np.float32).reshape(-1)
        tensor_blocks = []
        tensor_weight = 1.0 / host.size
        for start in range(0, host.size, BLOCK_SIZE):
            block = host[start : start + BLOCK_SIZE]
            scale = max(float(np.max(np.abs(block))) / 7.0, 1.0e-5)
            codes = np.clip(np.rint(block / scale) + ZERO_POINT, 0, 15)
            packed = pack_codes(codes)
            rows.append(packed)
            leaves.append(
                (element_offset, block.size, 4, scale, tensor_weight, scale)
            )
            tensor_blocks.append((byte_offset, block.size, scale))
            element_offset += block.size
            byte_offset += packed.size
        specs.append((value.shape, tuple(tensor_blocks)))
    return np.concatenate(rows).astype(np.uint8), leaves, tuple(specs)

def decode_params(row, specs):
    row = jnp.asarray(row, dtype=jnp.uint8)
    params = []
    for shape, blocks in specs:
        values = []
        for byte_offset, length, scale in blocks:
            byte_length = (length + 1) // 2
            packed = row[byte_offset : byte_offset + byte_length]
            codes = jnp.stack((packed & 0x0F, packed >> 4), axis=1).reshape(-1)
            values.append((codes[:length].astype(jnp.float32) - ZERO_POINT) * scale)
        params.append(jnp.concatenate(values).reshape(shape))
    return tuple(params)

def device_batch(search, trials):
    owners = []
    rows = []
    for pointer, size, device in search.device_batch(trials):
        memory = cp.cuda.UnownedMemory(pointer, size, search, device_id=device)
        packed = cp.ndarray(
            (size,), dtype=cp.uint8, memptr=cp.cuda.MemoryPointer(memory, 0)
        )
        owners.append(packed)
        rows.append(jax.dlpack.from_dlpack(packed))
    return jnp.stack(rows), owners

def decode_batch(rows, specs):
    return jax.vmap(lambda row: decode_params(row, specs))(rows)

base_row, leaves, specs = encode_params(float_params)
base_params = decode_params(base_row, specs)
base_reward, base_std = score_policy(base_params)
base_reward.block_until_ready()
print(
    f"packed_bytes={base_row.nbytes:,} leaves={len(leaves)} "
    f"quantized_reward={float(base_reward):.5f} std={float(base_std):.5f}"
)

## 7. Optimize the policy with ENNx

Rust TuRBO owns trust-region adaptation, acceptance, and restart. Its packed RAASP path targets 20 changed codes per proposal. Four resident policy trials are selected from independent eight-candidate pools and evaluated together by one vectorized MJX/JAX program.

In [ ]:
import time

HISTORY_CAPACITY = 8
BATCH_ARMS = 4
CANDIDATES = 8
ROUNDS = 32
NUM_PERT = 20

search = TurboSearch(
    base_row,
    float(base_reward),
    leaves,
    HISTORY_CAPACITY,
    backend="cuda",
    num_pert=NUM_PERT,
    max_pending=BATCH_ARMS,
)
seed_rng = np.random.default_rng(41)
best_reward = float(base_reward)
best_std = float(base_std)
best_row = jax.device_put(base_row, GPU)
records = []

for round_index in range(ROUNDS):
    seeds = seed_rng.integers(
        0,
        np.iinfo(np.uint64).max,
        size=(BATCH_ARMS, CANDIDATES),
        dtype=np.uint64,
    )
    neighbors = min(8, search.history_len)

    proposal_start = time.perf_counter()
    trials = search.ask_batch(
        seeds,
        neighbors,
        acquisition="thompson",
        seed=round_index,
    )
    candidate_rows, packed_owners = device_batch(search, trials)
    proposal_seconds = time.perf_counter() - proposal_start

    evaluation_start = time.perf_counter()
    reward, reward_std = score_batch(decode_batch(candidate_rows, specs))
    reward.block_until_ready()
    rewards = np.asarray(reward, dtype=np.float32)
    reward_stds = np.asarray(reward_std, dtype=np.float32)
    evaluation_seconds = time.perf_counter() - evaluation_start

    accepted = search.tell_batch(trials, rewards)
    best_index = int(np.argmax(rewards))
    if float(rewards[best_index]) > best_reward:
        best_reward = float(rewards[best_index])
        best_std = float(reward_stds[best_index])
        best_row = candidate_rows[best_index]

    for arm_index, trial in enumerate(trials):
        records.append(
            {
                "round": round_index,
                "arm": arm_index,
                "candidate": trial.index,
                "seed": trial.seed,
                "predicted": trial.score,
                "reward": float(rewards[arm_index]),
                "reward_std": float(reward_stds[arm_index]),
                "best_reward": best_reward,
                "accepted": accepted[arm_index],
                "length": trial.length,
                "probability": trial.probability,
                "proposal_ms": proposal_seconds * 1_000,
                "evaluation_ms": evaluation_seconds * 1_000,
            }
        )
    print(
        f"round={round_index:02d} rewards={rewards.round(5).tolist()} "
        f"accepted={accepted} best={best_reward:.5f} length={search.length:.5f}"
    )

In [ ]:
import matplotlib.pyplot as plt

rounds = np.asarray([record["round"] for record in records])
rewards = np.asarray([record["reward"] for record in records])
best_rewards = np.asarray([record["best_reward"] for record in records])
proposal_ms = np.asarray([record["proposal_ms"] for record in records])
evaluation_ms = np.asarray([record["evaluation_ms"] for record in records])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(rounds, rewards, "o", alpha=0.55, label="observed")
axes[0].plot(rounds, best_rewards, linewidth=2, label="incumbent")
axes[0].axhline(float(base_reward), color="black", linestyle="--", label="start")
axes[0].set(xlabel="round", ylabel="mean reward per step")
axes[0].legend()
axes[1].plot(rounds, proposal_ms, label="ENN proposal")
axes[1].plot(rounds, evaluation_ms, label="MJX evaluation")
axes[1].set(xlabel="round", ylabel="milliseconds")
axes[1].legend()
fig.tight_layout()

## 8. Render the optimized Humanoid

The final rollout is simulated with the selected quantized policy. MuJoCo renders the stored MJX states from the tracking camera and writes an MP4 alongside the inline player.

In [ ]:
def run_rollout(params, steps=512):
    reset = jax.jit(reset_env)
    advance = jax.jit(step_env)
    act = jax.jit(apply_policy)
    data = reset(jax.random.PRNGKey(101))
    trajectory = []
    total = 0.0
    for _ in range(steps):
        trajectory.append(
            (np.asarray(data.qpos), np.asarray(data.qvel), np.asarray(data.ctrl))
        )
        action = act(params, observe(data))
        data, reward, done = advance(data, action)
        total += float(reward)
        if bool(done):
            break
    return trajectory, total

def render_video(trajectory, path):
    render_every = 2
    renderer = mujoco.Renderer(mj_model, height=480, width=640)
    cpu_data = mujoco.MjData(mj_model)
    frames = []
    for qpos, qvel, ctrl in trajectory[::render_every]:
        cpu_data.qpos[:] = qpos
        cpu_data.qvel[:] = qvel
        cpu_data.ctrl[:] = ctrl
        mujoco.mj_forward(mj_model, cpu_data)
        renderer.update_scene(cpu_data, camera="side")
        frames.append(renderer.render())
    renderer.close()
    fps = 1.0 / (CTRL_DT * render_every)
    media.write_video(path, frames, fps=fps)
    return frames, fps

best_params = decode_params(best_row, specs)
trajectory, video_reward = run_rollout(best_params)
VIDEO_PATH = "/content/ennx_mjx_humanoid.mp4"
frames, fps = render_video(trajectory, VIDEO_PATH)
print(
    f"frames={len(frames)} rollout_reward={video_reward:.3f} "
    f"best_eval={best_reward:.5f} +- {best_std:.5f} path={VIDEO_PATH}"
)
media.show_video(frames, fps=fps, loop=False)

## 9. What this measures

This is a direct high-dimensional black-box control experiment. ENNx searches approximately 972,000 quantized policy parameters using only repeated scalar task rewards. The simulator and neural policy execute in JAX on the T4; the selected packed policy remains GPU-resident and enters JAX through DLPack. Only scalar rewards return to the Rust optimizer.